# 插入脉冲

Canonical workflow notebook for inserted-pulse lifecycle studies. 所有用户可变参数集中在下一格 `CONFIG`。


## 1. 用户配置


In [ ]:
# 所有工况变体只修改本单元
CONFIG = {
    "cell": "MIC",
    "output_name": "插入脉冲",
    "run_mode": "smoke",  # "smoke" / "study"
    "nominal_capacity_ah": 1175.0,
    "nominal_voltage_v": 3.2,
    "base_p_rate": 0.25,
    "temperature_c": 25.0,
    "initial_soc": None,
    "charge_cutoff_v": 3.65,
    "discharge_cutoff_v": 2.50,
    "rest_minutes": 30.0,
    "period_minutes": 0.5,
    "capacity_check_interval_cycles": 500,
    "capacity_check_p_rate": 0.25,
    "capacity_check_at_start": True,
    "adapt_pulse_windows_to_capacity": True,
    "postprocess_adjustments": {
        # "pulse_0p3p": {"capacity_scale": 1.01, "capacity_offset_ah": 0.0, "retention_scale": 1.0, "retention_offset": -0.01},
    },
    "datasets": [
        # {"cell": "MIC", "temperature_c": 25, "test_type": "脉冲", "kind": "processed", "require_unique": False},
    ],
    "modes": {
        "smoke": {
            "total_cycles": 1,
            "cycles_per_block": 1,
            "aging_t_factor": 1,
            "conditioning_t_factor": 1,
            "showprogress": False,
            "return_solutions": True,
            "parallel": False,
        },
        "study": {
            "total_cycles": 10000,
            "cycles_per_block": 10,
            "aging_t_factor": 50,
            "conditioning_t_factor": 1,
            "showprogress": True,
            "return_solutions": True,
            "parallel": False,
            "max_workers": 2,
        },
    },
    "var_pts": {"x_n": 5, "x_s": 5, "x_p": 5, "r_n": 20, "r_p": 20},
    "scenarios": [
        {"name": "baseline_0p25p", "display_name": "0.25P 基础充放循环", "pulse_p_rate": None, "pulse_seconds": None, "charge_interval_minutes": None, "discharge_interval_minutes": None, "charge_soc_window": (0.10, 0.90), "discharge_soc_window": (0.10, 0.90), "enabled": True},
        {"name": "pulse_0p3p", "display_name": "0.3P 插入脉冲 2 min", "pulse_p_rate": 0.3, "pulse_seconds": 120.0, "charge_interval_minutes": 5.0, "discharge_interval_minutes": 5.0, "charge_soc_window": (0.10, 0.93), "discharge_soc_window": (0.10, 0.90), "enabled": True},
        {"name": "pulse_0p375p", "display_name": "0.375P 插入脉冲 60 s", "pulse_p_rate": 0.375, "pulse_seconds": 60.0, "charge_interval_minutes": 30.0, "discharge_interval_minutes": 20.0, "charge_soc_window": (0.10, 0.76), "discharge_soc_window": (0.29, 1.00), "enabled": True},
        {"name": "pulse_0p75p", "display_name": "0.75P 插入脉冲 13 s", "pulse_p_rate": 0.75, "pulse_seconds": 13.0, "charge_interval_minutes": 30.0, "discharge_interval_minutes": 20.0, "charge_soc_window": (0.10, 0.76), "discharge_soc_window": (0.30, 1.00), "enabled": True},
    ],
}
ANALYSIS = {"capacity": True, "voltage_diagnostics": True, "export": True}


## 2. 环境与导入


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

import matplotlib.pyplot as plt
plt.style.use("science")
plt.rcParams["font.family"] = "Calibri, Microsoft YaHei"
import numpy as np
import pandas as pd

SEARCH_ROOT = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in (SEARCH_ROOT, *SEARCH_ROOT.parents):
    if (candidate / "src" / "easy_imports.py").exists() and (candidate / "pyproject.toml").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"Cannot locate BatteryProject root from {SEARCH_ROOT}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.easy_imports import configure_notebook_environment, extract_cycle_voltage_curve
from src.workflows.pulse import PulseWorkflowSpec, build_adjusted_capacity_table, prepare_pulse_workflow, run_pulse_workflow

PROJECT_ROOT, WORKSPACE_ROOT, PARAMS_ROOT = configure_notebook_environment(project_root=PROJECT_ROOT, include_workspace_root=True)


## 3. 配置校验与数据查询


In [ ]:
spec = PulseWorkflowSpec.from_mapping(CONFIG)
prepared = prepare_pulse_workflow(spec)
dataset_entries = []
for query in spec.datasets:
    dataset_entries.extend(query.resolve(WORKSPACE_ROOT))
print(f"run_mode={spec.run_mode}; scenarios={len(prepared['scenarios'])}; datasets={len(dataset_entries)}")
prepared["scenario_table"]


## 4. 寿命仿真


In [ ]:
result = run_pulse_workflow(spec, project_root=PROJECT_ROOT, workspace_root=WORKSPACE_ROOT)
results = result["results"]
summary_df = result["summary_df"]
capacity_check_df = result["capacity_check_df"]
print("输出目录:", result["context"].run_dir)
summary_df


## 5. 容量保持率后处理与导出


In [ ]:
if ANALYSIS["capacity"]:
    capacity_check_adjusted_df = build_adjusted_capacity_table(results, spec.postprocess_adjustments)
    if not capacity_check_adjusted_df.empty:
        fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2), constrained_layout=False)
        for label, frame in capacity_check_adjusted_df.groupby("scenario"):
            axes[0].plot(frame["real_cycle"], frame["capacity_retention_adjusted"] * 100.0, marker="o", label=label)
            axes[1].plot(frame["real_cycle"], frame["capacity_check_discharge_capacity_ah_adjusted"], marker="o", label=label)
        axes[0].set_xlabel("Cycle number")
        axes[0].set_ylabel("Capacity retention [%]")
        axes[0].set_title("Capacity retention from periodic normal discharge", pad=10)
        axes[0].ticklabel_format(axis="y", style="plain", useOffset=False)
        axes[0].grid(True, alpha=0.3)
        axes[1].set_xlabel("Cycle number")
        axes[1].set_ylabel("Normal discharge capacity [Ah]")
        axes[1].set_title("Periodic normal discharge capacity", pad=10)
        axes[1].ticklabel_format(axis="y", style="plain", useOffset=False)
        axes[1].grid(True, alpha=0.3)
        handles, labels = axes[0].get_legend_handles_labels()
        fig.subplots_adjust(top=0.70, bottom=0.15, wspace=0.24)
        fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.98), ncol=2, frameon=False)
        plot_path = result["context"].plots_dir / "capacity_retention.png"
        fig.savefig(plot_path, dpi=180)
        adjusted_path = result["context"].artifacts_dir / "capacity_check_adjusted.csv"
        capacity_check_adjusted_df.to_csv(adjusted_path, index=False)
        print("容量图:", plot_path)
        print("修正后容量表:", adjusted_path)
    else:
        print("无 capacity check 数据可绘图。")


## 6. 电压曲线检查


In [ ]:
if ANALYSIS["voltage_diagnostics"]:
    voltage_scenarios = [scenario["name"] for scenario in spec.scenarios if scenario.get("pulse_p_rate") is not None]
    if voltage_scenarios:
        fig, axes = plt.subplots(len(voltage_scenarios), 2, figsize=(13, 4 * len(voltage_scenarios)), constrained_layout=True)
        axes = np.atleast_2d(axes)
        max_cycle = max((max(bundle.get("solution_real_cycles", [1])) for bundle in results.values() if bundle.get("solution_real_cycles")), default=1)
        cmap = plt.get_cmap("coolwarm")
        norm = plt.Normalize(vmin=1, vmax=max_cycle)
        for row, scenario_name in enumerate(voltage_scenarios):
            bundle = results.get(scenario_name, {})
            solutions = bundle.get("solutions", [])
            real_cycles = bundle.get("solution_real_cycles", [])
            for sol, real_cycle in zip(solutions, real_cycles):
                cycle = sol.cycles[-1] if getattr(sol, "cycles", None) else sol
                color = cmap(norm(real_cycle))
                charge = extract_cycle_voltage_curve(cycle, direction="charge", x_axis="capacity")
                discharge = extract_cycle_voltage_curve(cycle, direction="discharge", x_axis="time")
                if charge["x"].size:
                    axes[row, 0].plot(charge["x"], charge["voltage"], color=color, linewidth=1.2)
                if discharge["x"].size:
                    axes[row, 1].plot(discharge["x"], discharge["voltage"], color=color, linewidth=1.2)
            label = bundle.get("scenario", {}).get("display_name", scenario_name)
            axes[row, 0].set_title(f"{label}: charge voltage")
            axes[row, 0].set_xlabel("Capacity [Ah]")
            axes[row, 0].set_ylabel("Voltage [V]")
            axes[row, 0].grid(True, alpha=0.3)
            axes[row, 1].set_title(f"{label}: discharge voltage")
            axes[row, 1].set_xlabel("Time [s]")
            axes[row, 1].set_ylabel("Voltage [V]")
            axes[row, 1].grid(True, alpha=0.3)
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        fig.colorbar(sm, ax=axes.ravel().tolist(), label="Cycle number")
        voltage_path = result["context"].plots_dir / "voltage_diagnostics.png"
        fig.savefig(voltage_path, dpi=180)
        print("电压诊断图:", voltage_path)
    else:
        print("无插入脉冲场景，跳过电压诊断。")


## 7. Artifact 链接


In [ ]:
pd.DataFrame([{"artifact": key, "path": str(path)} for key, path in result["artifact_paths"].items()])
